In [50]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [51]:
import shutil
import json
from pathlib import Path

import numpy as np

from standes.utils import generate_type_1_tag


from phd_project.scripts.templates.copy_templates_to_folders import (
    # copy_pushover_config,
    copy_analysis_config,
    copy_structural_model,
    configure_batch_run_file,
    copy_file,
)

from phd_project.config import config

cfg = config.load_config()

In [52]:
def get_control_node_from_design_file(design_file_path: Path) -> int:
    # roof node
    with open(design_file_path, "r") as f:
        design_data = json.load(f)
    
    n_levels = len(design_data["structure"]["level_coordinates"])
    # node tag for top left corner of the structure
    tag = generate_type_1_tag(1, 1, 1, n_levels, 0, 0)
    return tag

def get_n_damping_modes_from_design_file(design_file_path: Path) -> int:
    # the number of damping models is equal to the number of primary grid nodes with mass
    # this damps all horizontal modes of the structure
    n_damping_modes = get_n_modes_from_design_file(design_file_path) * 4
    return n_damping_modes


def get_n_modes_from_design_file(design_file_path: Path) -> int:
    # number of modes is equal to the number of levels in the structure
    with open(design_file_path, "r") as f:
        design_data = json.load(f)
    
    # the number of damping models is equal to the number of primary grid nodes with mass
    # this damps all horizontal modes of the structure
    n_modes = len(design_data["structure"]["level_coordinates"]) - 1
    return n_modes
    

In [53]:
# INPUTS

# design_file_root = cfg["models"]["casestudy_designs_ec8_gen2"]
analysis_root_folder = Path(r"E:\02_wp1pt4pt1_po_analyses_for_sdof_fitting\fitB_optimisation")

batch_run_filename = "sodf_fitB_parameter_slices"
# modal_batch_run_filename = "modal_analyses"
# nltha_batch_run_filename = "nltha_analyses"

# building_model_tags = ["3s_cbf_dc2_41"]

# # PO Parameters
# max_drift = 2       # in %; roof drift limit for the pushover analysis.
# drift_step = 0.005   # in %; the step size for the pushover analysis.

# # NLTHA Parameters
# gm_json_src_str = 'E:/gm_records_p695'

In [54]:
# building_tags = [f.name for f in design_file_root.iterdir() 
#                  if f.is_dir() and (f / f"{f.name}_out.json").exists()]
beta_values = [0, 0.5, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4]
damage2_values = np.arange(0, 20) / 10
building_tags = ["test_folder"]

In [ ]:
models_to_batchrun = []

for beta in beta_values:
    for d2 in damage2_values:

        building_tag = f"beta_{int(beta*100)}_d2_{int(d2*10)}"

        mat_2_args = [1, 1, 0, float(d2), float(beta)]

        # create the folder and files for each building model
        building_analysis_folder = analysis_root_folder / building_tag
        building_analysis_folder.mkdir(parents=True, exist_ok=True)

        ###### Structural Model File
        model_src = Path(r"E:\02_wp1pt4pt1_po_analyses_for_sdof_fitting\3s_cbf_dc2_41_fitB_sdof") / "structural_model.py"
        model_dst = building_analysis_folder / "structural_model.py"

        copy_structural_model(model_src, model_dst, ops_updates={"material_args_2": mat_2_args})

        # copy the materials
        

        ###### Pushover Files
        # copy the run po analysis file
        po_analysis_src = Path(r"E:\02_wp1pt4pt1_po_analyses_for_sdof_fitting\3s_cbf_dc2_41_fitB_sdof") / "run_cyclic_pushover.py"
        po_analysis_dst = building_analysis_folder / "run_cyclic_pushover.py"

        copy_file(po_analysis_src, po_analysis_dst)

        # create the po analysis config file
        po_config_src = Path(r"E:\02_wp1pt4pt1_po_analyses_for_sdof_fitting\3s_cbf_dc2_41_fitB_sdof") / "config_cyclic_pushover.py"
        po_config_dst = building_analysis_folder / "config_cyclic_pushover_1.py"

        po_outfolder = f"cyclic_pushover_1"
        config_updates = None

        copy_analysis_config(po_config_src, po_config_dst, 
                             results_folder_name=po_outfolder, update_config=config_updates)


        # add to scripts and configs list for batch file run
        models_to_batchrun.append({
                "script": po_analysis_dst,
                "config": [po_config_dst]
            })

    # ###### NLTHA Files
    # # copy the run nltha analysis file
    # nltha_analysis_src = cfg["templates"]["run_nltha_ud"]
    # nltha_analysis_dst = building_analysis_folder / "run_nltha_ud.py"

    # copy_file(nltha_analysis_src, nltha_analysis_dst)

    # # create the nltha config file
    # nltha_config_src = cfg["templates"]["config_nltha"]
    # for record, sf in [
    #     ("fema_p695_120121.json", 2.0), 
    #     ("fema_p695_120621.json", 2.6)]: 
        
    #     rec_num = record.split(".json")[0].split("_")[-1]

    #     nltha_config_dst = building_analysis_folder / f"config_nltha_{rec_num}.py"

    #     nltha_outfolder = f"nltha_{rec_num}_sf{int(sf*1000)}"
    #     config_updates = {
    #         'gm_json_file': record,
    #         'scale_factor': sf
    #     }

    #     copy_analysis_config(nltha_config_src, nltha_config_dst, 
    #                          results_folder_name=nltha_outfolder, 
    #                          update_config=config_updates,
    #                          gm_json_src_str=gm_json_src_str)
        
    #     # add to scripts and configs list for batch file run
    #     if building_tag.split("_")[2] == "dc0":
    #         nltha_scripts_configs_list_dc0.append({
    #             "script": nltha_analysis_dst,
    #             "config": [nltha_config_dst]
    #         })
    #     elif building_tag.split("_")[2] == "dc1":
    #         nltha_scripts_configs_list_dc1.append({
    #             "script": nltha_analysis_dst,
    #             "config": [nltha_config_dst]
    #         })
    #     elif building_tag.split("_")[2] == "dc2":
    #         nltha_scripts_configs_list_dc2.append({
    #             "script": nltha_analysis_dst,
    #             "config": [nltha_config_dst]
    #         })
    #     elif building_tag.split("_")[2] == "dc3":
    #         nltha_scripts_configs_list_dc3.append({
    #             "script": nltha_analysis_dst,
    #             "config": [nltha_config_dst]
    #         })

    # ###### Modal Analysis Files
    # # copy the run modal analysis file
    # modal_analysis_src = cfg["templates"]["run_modal"]
    # modal_analysis_dst = building_analysis_folder / "run_modal.py"

    # copy_file(modal_analysis_src, modal_analysis_dst)

    # # create the modal analysis config file
    # modal_config_src = cfg["templates"]["config_modal"]
    # modal_config_dst = building_analysis_folder / "config_modal.py"

    # ctrl_node = get_control_node_from_design_file(building_design_file_dst)
    # Umax = ("drift", max_drift)
    # dU = drift_step

    # modal_outfolder = f"modal"
    # config_updates = {
    #     "n_modes": get_n_modes_from_design_file(building_design_file_dst)
    # }

    # copy_analysis_config(modal_config_src, modal_config_dst, 
    #                      results_folder_name=modal_outfolder, 
    #                      update_config=config_updates)

    # # add to scripts and configs list for batch file run
    # if building_tag.split("_")[2] == "dc0":
    #     modal_scripts_configs_list_dc0.append({
    #         "script": modal_analysis_dst,
    #         "config": [modal_config_dst]
    #     })
    # elif building_tag.split("_")[2] == "dc1":
    #     modal_scripts_configs_list_dc1.append({
    #         "script": modal_analysis_dst,
    #         "config": [modal_config_dst]
    #     })
    # elif building_tag.split("_")[2] == "dc2":
    #     modal_scripts_configs_list_dc2.append({
    #         "script": modal_analysis_dst,
    #         "config": [modal_config_dst]
    #     })
    # elif building_tag.split("_")[2] == "dc3":
    #     modal_scripts_configs_list_dc3.append({
    #         "script": modal_analysis_dst,
    #         "config": [modal_config_dst]
    #     })



In [56]:
# create a script for batch running the analyses
batch_run_src = cfg["templates"]["batch_run"]

batch_run_dst = cfg["scripts"]["wp1pt4pt1_batch_run"] / f"{batch_run_filename}.py"
configure_batch_run_file(batch_run_src, batch_run_dst, models_to_batchrun)
